In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
from numpy.core.numeric import infty

sns.set_style("whitegrid")


In [ ]:
# models : "BPR", "VAE", "NCF", "SVD"

legend_plot = 'NCF/general_results/NCF_Yelp_NDCG_test_LDP.pdf'  #for which plot we want to have legend, this is the saving location of that image
paths = ["BPR", "VAE", "NCF", "SVD"]  # all models

In [ ]:
common_metrics = [
    'KLD_test', 'NDCG_test', 'recall_test', 'Total MRR',
    'normal_PL', 'novelty_normal',
    'Coverage', 'normalized_DPF',
    'normalized_exposure_1', 'normalized_exposure_2'
]

model_configs = {
    'BPR': {
        'csv_files': [
            'BPR_1M_LDP.csv', 'BPR_1M_DPSGD.csv',  # csv files that include our results
            'BPR_Yelp_LDP.csv', 'BPR_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["1M_BPR_50", "BPR_YELP"],
        'normal_names': ['1M_BPR_50', "", 'BPR_YELP'],
        'model_labels': ['BPR_1M', "", 'BPR_YELP'],
        'metrics': common_metrics
    },
    'SVD': {
        'csv_files': [
            'SVD_1M_LDP.csv', 'SVD_1M_DPSGD.csv',
            'SVD_Yelp_LDP.csv', 'SVD_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["1M_SVD_50", "SVD_Yelp_50"],
        'normal_names': ['1M_SVD_50', "", 'SVD_Yelp_50'],
        'model_labels': ['SVD_1M', "", 'SVD_Yelp'],
        'metrics': common_metrics
    },
    'VAE': {
        'csv_files': [
            'VAE_1M_LDP.csv', 'VAE_1M_DPSGD.csv',
            'VAE_Yelp_LDP.csv', 'VAE_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["vae_1M", "vae_Yelp"],
        'normal_names': ['vae_1M', "", 'vae_Yelp'],
        'model_labels': ['VAE_1M', "", 'VAE_Yelp'],
        'metrics': [m for m in common_metrics if m != 'Total MRR']
    },
    'NCF': {
        'csv_files': [
            'Deep_1M_LDP.csv', 'Deep_1M_DPSGD.csv',
            'Deep_Yelp_LDP.csv', 'Deep_Yelp_DPSGD.csv'
        ],
        'normal_model_name': ["Deep_1M_50", 'Yelp_Deep_50'],
        'normal_names': ['Deep_1M_50', "", 'Yelp_Deep_50'],
        'model_labels': ['NCF_1M', "", 'NCF_Yelp'],
        'metrics': common_metrics
    }
}

In [ ]:
# defines the scales (range) of each metric (this scale will be used to have the same scale for relevant plots, to make them comparable)
def scale_deter(df1, df2, metric, normal):
    df1.replace('', np.nan, inplace=True)
    df2.replace('', np.nan, inplace=True)
    df1.replace(' ', np.nan, inplace=True)
    df2.replace(' ', np.nan, inplace=True)
    df1.replace(['N/A', 'NA', 'None'], np.nan, inplace=True)
    df2.replace(['N/A', 'NA', 'None'], np.nan, inplace=True)
    df1[metric] = pd.to_numeric(df1[metric], errors='coerce')
    df2[metric] = pd.to_numeric(df2[metric], errors='coerce')
    df1 = df1.dropna(subset=[metric])
    df2 = df2.dropna(subset=[metric])
    ndcg_noise_df2 = df2.groupby('noise_multiplier')[metric].agg(['mean', 'std']).reset_index()  # based on DPSGD
    ndcg_noise_df1 = df1.groupby('epsilon')[metric].agg(['mean', 'std']).reset_index()  # based on LDP

    y_min = min(ndcg_noise_df1['mean'].min() - ndcg_noise_df1['std'].min(),
                ndcg_noise_df2['mean'].min() - ndcg_noise_df2['std'].min(), np.mean(normal[metric]))
    y_max = max(ndcg_noise_df1['mean'].max() + ndcg_noise_df1['std'].max(),
                ndcg_noise_df2['mean'].max() + ndcg_noise_df2['std'].max(), np.mean(normal[metric]))

    print(f"{metric}:y_min: {y_min}, y_max: {y_max}")
    return y_min, y_max


In [ ]:
def scale_deter(df1, df2, metric, normal):
    ndcg_noise_df2 = df2.groupby('noise_multiplier')[metric].agg(['mean', 'std']).reset_index()
    ndcg_noise_df1 = df1.groupby('epsilon')[metric].agg(['mean', 'std']).reset_index()

    y_min = min(ndcg_noise_df1['mean'].min() - ndcg_noise_df1['std'].min(),
                ndcg_noise_df2['mean'].min() - ndcg_noise_df2['std'].min(), np.mean(normal[metric]))
    y_max = max(ndcg_noise_df1['mean'].max() + ndcg_noise_df1['std'].max(),
                ndcg_noise_df2['mean'].max() + ndcg_noise_df2['std'].max(), np.mean(normal[metric]))

    print(f"{metric}:y_min: {y_min}, y_max: {y_max}")
    return y_min, y_max

In [ ]:
def plot_metrics_epsilon_based(df1, df2, normal, metric, title, scale, cleaned_metrics_name, path_save_base,
                               legend_plot):
    ndcg_normal_mean = normal[metric].mean()

    fig1, ax1 = plt.subplots(figsize=(3, 2.5))
    ax1.set_facecolor('white')
    ax1.set_xlim(0.008, 406)  # min/max epsilon over all datasets as of 2025-07-17

    sns.lineplot(
        data=df2[[metric, 'epsilon']],
        x="epsilon", y=metric,
        errorbar=('ci', 95), err_style='bars', err_kws={'fmt': '.', 'ecolor': 'black'},
        color=sns.color_palette("Paired")[1],
        ax=ax1, label=f'DPSGD'
    )
    sns.lineplot(
        data=df1[[metric, 'epsilon']],
        x="epsilon", y=metric,
        errorbar=('ci', 95), err_style='bars', err_kws={'fmt': '.', 'ecolor': 'black'},
        color=sns.color_palette("Paired")[3],
        ax=ax1, label=f'LDP'
    )

    ax1.set_xlabel('Epsilon')
    ax1.set_ylabel(cleaned_metrics_name[metric])
    ax1.axhline(y=ndcg_normal_mean, color='red', linewidth=1, linestyle='--', label=f'no privacy')
    ax1.grid(True)
    ax1.legend()
    if title != "NCF_1M":
        ax1.get_legend().remove()

    ax1.set_xscale('log', base=10)
    ax1.set_ylim(scale[0] - 0.1, scale[1] + 0.1)
    locmin = mticker.LogLocator(base=10, subs=np.arange(0.1, 1, 0.1), numticks=10)
    ax1.xaxis.set_minor_locator(locmin)

    sns.despine()
    plt.tight_layout()
    plt.savefig(path_save_base, format='pdf', bbox_inches='tight', dpi=300)
    plt.show()
    plt.close(fig1)

    # path_save_ax2 = path_save_base.replace(".pdf", "_LDP.pdf")
    # fig2, ax2 = plt.subplots(figsize=(3, 2.5))
    # ax2.set_facecolor('white')
    #
    # sns.lineplot(
    #     data=df1[[metric, 'epsilon']],
    #     x="epsilon", y=metric,
    #     errorbar=('ci', 95), err_style='bars', err_kws={'fmt': '.', 'ecolor': 'black'},
    #     ax=ax2, label=f'with privacy'
    # )
    #
    # ax2.set_xlabel('Epsilon')
    # ax2.set_ylabel(cleaned_metrics_name[metric])
    # ax2.set_title(f"{title} with LDP")
    # ax2.axhline(y=ndcg_normal_mean, color='red', linewidth=1, linestyle='--', label=f'without privacy')
    # ax2.grid(True)
    # ax2.legend()
    # if path_save_ax2 != legend_plot:
    #     ax2.get_legend().remove()
    # ax2.set_xscale('log', base=10)
    # ax2.set_ylim(scale[0] - 0.1, scale[1] + 0.1)
    #
    # sns.despine()
    # plt.tight_layout()
    # plt.savefig(path_save_ax2, format='pdf', bbox_inches='tight', dpi=300)
    # plt.show()
    # plt.close(fig2)


In [ ]:
cleaned_metrics_name = {'KLD_test': 'KLD', 'NDCG_test': 'NDCG', 'recall_test': 'Recall', 'Total MRR': 'MRR',
                        'normal_PL': 'PL', 'novelty_normal': 'Novelty',
                        'Coverage': 'Coverage', 'normalized_DPF': 'DPF',
                        'normalized_exposure_1': 'Exposure_1', 'normalized_exposure_2': 'Exposure_2'}

for path in paths:
    dataset = model_configs[path]
    df_dpsgd = pd.read_csv(os.path.join('csv_files', dataset['csv_files'][1]))
    df_dpsgd_yelp = pd.read_csv(os.path.join('csv_files', dataset['csv_files'][3]))
    df_normal_1M = df_dpsgd.loc[df_dpsgd['model_name'] == dataset['normal_model_name'][0]]
    df_normal_yelp = df_dpsgd_yelp.loc[df_dpsgd_yelp['model_name'] == dataset['normal_model_name'][1]]
    # this is the same for all models
    normal_models = [df_normal_1M, "", df_normal_yelp]
    metrics = ['KLD_test', 'NDCG_test', 'recall_test',
               'normal_PL', 'novelty_normal',
               'Coverage', 'normalized_DPF',
               'normalized_exposure_1', 'normalized_exposure_2']

    print("minimum epsilon:", min([x for x in df_dpsgd['epsilon'] if x > 0]))
    print("minimum epsilon:", min([x for x in df_dpsgd_yelp['epsilon'] if x > 0]))
    print("maximum epsilon:", max([x for x in df_dpsgd['epsilon'] if x < infty]))
    print("maximum epsilon:", max([x for x in df_dpsgd_yelp['epsilon'] if x < infty]))

    scales = {}
    for i, metric in enumerate(dataset['metrics']):
        y_max = -100
        y_min = float(
            "inf")  # the biggest possible value: this ensures that, y_min will be replaced with min calculated by scale_deter
        for i in range(0, len(dataset['csv_files']), 2):  # go through all csv files for the model
            df_mechanism1 = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][i]))
            df_mechanism1 = df_mechanism1[df_mechanism1['epsilon'] > 0]
            df_mechanism2 = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][i + 1]))
            df_mechanism2 = df_mechanism2[df_mechanism2['epsilon'] > 0]
            normal = normal_models[i]
            print(f"inside: {metric}:y_min: {y_min}, y_max: {y_max}")
            x_min_init, x_max_init = scale_deter(df_mechanism1, df_mechanism2, metric, normal)
            if x_min_init < y_min:
                y_min = x_min_init
            if x_max_init > y_max:
                y_max = x_max_init
        print(f"outside: {metric}:y_min: {y_min}, y_max: {y_max}")
        scales[metric] = [y_min, y_max]
        print(f"scales: {scales}")
    if path == 'BPR':
        name = 'BPR'
    elif path == 'NCF':
        name = 'NCF'
    if path == 'SVD':
        name = 'SVD'
    if path == 'VAE':
        name = 'VAE'

    print(path)
    new_path = os.path.join(path, 'general_results')  # save the plots in this directory
    if not os.path.exists(new_path):
        os.makedirs(new_path)

    for i, metric in enumerate(metrics):
        for i in range(0, len(dataset['csv_files']), 2):
            df_mechanism1 = pd.read_csv(os.path.join("csv_files", dataset['csv_files'][i]))
            df_mechanism1 = df_mechanism1[df_mechanism1['epsilon'] > 0]
            df_mechanism2 = pd.read_csv(os.path.join("csv_files", dataset["csv_files"][i + 1]))
            df_mechanism2 = df_mechanism2[df_mechanism2['epsilon'] > 0]

            model_name = dataset['model_labels'][i]
            normal = normal_models[i]
            file_name = f'{model_name}_{metric}.pdf'
            path_save = os.path.join(new_path, file_name)

            plot_metrics_epsilon_based(df_mechanism1, df_mechanism2, normal, metric, model_name, scales[metric],
                                       cleaned_metrics_name, path_save, legend_plot)
